# Barrido ε-constraint: **min costo** s.a. carbono ≥ ε
Cada punto minimiza el costo con una meta mínima de carbono ε. ε recorre `[carbono del ancla A, carbono del ancla B]` en `N_PUNTOS` intervalos iguales, de lo más laxo (ancla A) a lo más estricto (ancla B).

Formulación AUGMECON: `min costo − ρ·r_costo·s/r_carbono`  s.a.  `carbono − s = ε`, `s ≥ 0`.

**Cómo se elige ε (no es arbitrario):** primero se calcula la tabla de pagos con dos anclas lexicográficas
- **A:** min costo → `C_min`; luego max carbono con `costo ≤ C_min`.
- **B:** max carbono → `G_max`; luego min costo con `carbono ≥ G_max`.

La malla de ε divide el rango entre A y B; los extremos de la malla son las propias anclas.
Las anclas se guardan en `resultados_<modo>/anclas/anclas.pkl` y las reutiliza el otro notebook (`barrido_maxCarbono.ipynb`) si los datos y parámetros no cambiaron.

**AUGMECON2:** el término aumentado evita puntos débilmente eficientes; el *bypass* salta los ε que darían la misma solución.
Cada punto arranca (warm start) desde la mejor solución ya calculada que cumple su ε.

**Salidas** (en `resultados_<modo>/`, con `<modo>` = `selLibre` o `selFija`):
- `frontera_minCosto.csv`: un renglón por punto con ε, costo, carbono, holgura, estado, gap, tiempo, dominado y resumen del plan.
- `barrido_minCosto/minCosto_pXX_calendario.csv`, `_compras.csv`, `_plan_siembra.csv`: plan completo de cada punto.
- `frontera_combinada.csv`: se genera cuando ya existen los dos barridos.

In [1]:
import importlib
import pandas as pd
from IPython.display import display
import modelo_base as mb
importlib.reload(mb)          # para tomar cambios de modelo_base.py sin reiniciar el kernel

<module 'modelo_base' from 'c:\\Users\\REPO\\Tec2026\\Cadena Suministro Reforestacion\\Version4-sumaponderada\\modelo_base.py'>

## Configuración
Todo lo editable vive en `mb.cfg`. El modo (`selFija` o `selLibre`) se cambia **solo en `modelo_base.py`** (`Config.LIBERAR_SELECCION`), para que ambos notebooks usen siempre la misma carpeta de resultados. Después de cambiarlo, reinicia el kernel de los dos notebooks.

In [2]:
# mb.cfg.CARPETA = r"C:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Modelo3"   # por defecto: carpeta de modelo_base.py
# El modo (selFija / selLibre) se define en modelo_base.py -> Config.LIBERAR_SELECCION
print('Modo de la corrida:', mb.cfg.MODO, '| resultados en', mb.cfg.CARPETA_RESULTADOS)
mb.cfg.N_PUNTOS     = 10      # intervalos de la malla de ε
mb.cfg.TIEMPO_FIJA  = 120     # fase 1 de cada resolución: selección de polígonos fija
mb.cfg.TIEMPO_PUNTO = 300     # fase 2 (solo selLibre): selección libre
mb.cfg.GAP          = 0.01
mb.cfg.RHO          = 1e-3
mb.cfg.USAR_BYPASS  = True

# Decisión a posteriori sobre la frontera
mb.cfg.PESO_CARBONO   = 0.5     # w del carbono (1 - w para el costo)
mb.cfg.PRECIO_CARBONO = None    # $ por unidad de carbono; None = no se usa

Modo de la corrida: selFija | resultados en c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija


In [3]:
datos = mb.cargar_datos()
problemas = mb.diagnostico(datos)

Adyacencia: 94 filas en el CSV -> 48 pares únicos
Modo: selFija | resultados en c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija
Capacidad: hasta 4.58 ha/día; mínimo exigido 4.0 ha/día
Se necesitan entre 108 y 124 ha (hay 189.5 ha) | compras con llegada entre el día 1 y el 27
Chequeo previo: sin problemas evidentes.


In [4]:
model = mb.construir_modelo(datos)

## Tabla de pagos (anclas A y B)
Si ya existen anclas guardadas con los mismos datos y parámetros, se cargan; si no, se calculan (etapa 0 → costo mínimo → anclas). Para forzar el recálculo usa `recalcular=True`.

In [5]:
anclas = mb.obtener_anclas(model, datos, recalcular=False)

Anclas cargadas de c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija\anclas\anclas.pkl (mismos datos y parámetros).
Tabla de pagos (selFija): rango costo = 222,157.5 | rango carbono = 156,790.1


,ancla,costo,carbono,estado,gap
0,"A (min costo, luego max carbono)",2424629.0,798767.321014,maxTimeLimit,0.06028
1,"B (max carbono, luego min costo)",2646786.5,955557.431871,optimal,0.00827


## Barrido `minCosto`

In [6]:
frontera = mb.barrido(model, anclas, "minCosto")
if frontera is not None:
    cols = ["punto", "epsilon", "costo", "carbono", "holgura", "estado", "gap", "segundos",
            "dominado", "duplicado", "n_poligonos", "ha_total", "plantado", "edad_promedio", "nota"]
    display(frontera[[c for c in cols if c in frontera.columns]])

Barrido minCosto | modo selFija | resultados en c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija
18:07:48 inicia minCosto p01
[minCosto p01 ε=814,446.3 [fase 1: selección fija]] optimal | costo = 2,415,059.0 | carbono = 823,216.8 | gap = 0.34% | 104 s
18:09:32 inicia minCosto p02
load_solution=False and check results.termination_condition and
results.found_feasible_solution() before loading a solution.
[minCosto p02 ε=830,125.3 [fase 1: selección fija]] maxTimeLimit | costo = 2,440,947.0 | carbono = 831,741.8 | gap = 1.33% | 128 s
18:11:41 inicia minCosto p03
load_solution=False and check results.termination_condition and
results.found_feasible_solution() before loading a solution.
[minCosto p03 ε=845,804.4 [fase 1: selección fija]] maxTimeLimit | costo = 2,460,588.5 | carbono = 845,828.0 | gap = 1.56% | 127 s
18:13:48 inicia minCosto p04
load_solution=False and check results.termination_condition and
results.found_feasible_solution() befo

,punto,epsilon,costo,carbono,holgura,estado,gap,segundos,dominado,duplicado,n_poligonos,ha_total,plantado,edad_promedio,nota
0,p00,798767.321014,2424629.0,798767.321014,NaN,maxTimeLimit,0.060280,126.8,True,False,17,115.45,54528.0,4.108825,ancla A
1,p01,814446.332100,2415059.0,823216.773013,8770.440913,optimal,0.003389,103.9,False,False,17,115.45,54529.0,4.013461,
2,p02,830125.343185,2440947.0,831741.834103,1616.490917,maxTimeLimit,0.013297,127.6,False,False,17,115.45,54529.0,3.821196,
3,p03,845804.354271,2460588.5,845827.959190,23.604918,maxTimeLimit,0.015616,126.8,False,False,17,115.45,54767.0,3.790896,
4,p04,861483.365357,2478996.0,861488.093695,4.728338,maxTimeLimit,0.016837,127.2,True,False,17,115.45,55071.0,3.776670,
5,p05,877162.376443,2476928.5,877167.764843,5.388400,optimal,0.009812,111.7,False,False,17,115.45,55389.0,3.812869,
6,p06,892841.387528,2487781.5,892842.600226,1.212697,optimal,0.007841,44.6,False,False,17,115.45,55718.0,3.840967,
7,p07,908520.398614,2515684.0,908520.457754,0.059140,optimal,0.009992,90.2,False,False,17,115.45,56025.0,3.805569,
8,p08,924199.409700,2573364.5,924207.042948,7.633248,maxTimeLimit,0.032727,955.1,False,False,17,115.45,56724.0,3.681387,
9,p09,939878.420786,2589229.5,939882.563504,4.142719,optimal,0.009999,99.4,False,False,17,115.45,57554.0,3.760138,


## Frontera combinada
Une ambos barridos, quita dominados y duplicados, y calcula el costo marginal por unidad de carbono y el punto rodilla. Requiere haber corrido también el otro notebook en el mismo modo.

In [7]:
combinada = mb.combinar_fronteras()
if combinada is not None:
    display(combinada[["barrido", "punto", "costo", "carbono", "costo_marginal", "rodilla", "estado", "gap"]])

Combinando fronteras del modo selFija en c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija
Guardado: c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija\frontera_combinada.csv (9 puntos no dominados)


,barrido,punto,costo,carbono,costo_marginal,rodilla,estado,gap
0,minCosto,p01,2415059.0,823216.773013,NaN,False,optimal,0.003389
1,minCosto,p02,2440947.0,831741.834103,3.036694,False,maxTimeLimit,0.013297
2,minCosto,p03,2460588.5,845827.959190,1.394386,False,maxTimeLimit,0.015616
3,minCosto,p05,2476928.5,877167.764843,0.521382,False,optimal,0.009812
4,minCosto,p06,2487781.5,892842.600226,0.692384,True,optimal,0.007841
5,minCosto,p07,2515684.0,908520.457754,1.779739,False,optimal,0.009992
6,minCosto,p08,2573364.5,924207.042948,3.677059,False,maxTimeLimit,0.032727
7,minCosto,p09,2589229.5,939882.563504,1.012088,False,optimal,0.009999
8,maxCarbono,p10,2646786.5,955557.431871,3.671929,False,optimal,0.008270


## Decisión a posteriori: elegir un plan de la frontera
Se evalúa la frontera **ya calculada** (no se vuelve a optimizar):
- **Suma ponderada normalizada:** `max w·z_carbono + (1−w)·z_costo`, con ambos objetivos escalados a [0, 1] usando las anclas.
- **Sensibilidad:** qué punto se elige para cada peso de 0 a 1. Solo los puntos *soportados* (envolvente convexa) pueden ganar.
- **TOPSIS** con los mismos pesos, como contraste.
- **Precio del carbono** (opcional): `max p·carbono − costo`, un peso con sentido económico.

In [8]:
resultado = mb.decision_a_posteriori()
if resultado is not None:
    decision, sensibilidad = resultado
    display(decision[["barrido", "punto", "costo", "carbono", "soportado", "score_ponderado", "topsis", "elegido_por"]])

Suma ponderada (w_carbono = 0.5, w_costo = 0.5): punto minCosto p06 | costo 2,487,781.5 | carbono 892,842.6
TOPSIS (mismos pesos): punto minCosto p09 | costo 2,589,229.5 | carbono 939,882.6
Puntos soportados (los únicos que la suma ponderada puede elegir): 5 de 9
Sensibilidad del peso del carbono:
  w_carbono en [0.00, 0.35] -> minCosto p01
  w_carbono en [0.40, 0.50] -> minCosto p06
  w_carbono en [0.55, 0.55] -> minCosto p07
  w_carbono en [0.60, 0.65] -> minCosto p09
  w_carbono en [0.70, 1.00] -> maxCarbono p10
Guardado: decision_selFija.csv y decision_sensibilidad_selFija.csv


,barrido,punto,costo,carbono,soportado,score_ponderado,topsis,elegido_por
0,minCosto,p01,2415059.0,823216.773013,True,0.500000,0.382756,
1,minCosto,p02,2440947.0,831741.834103,False,0.476350,0.371523,
2,minCosto,p03,2460588.5,845827.959190,False,0.487189,0.385953,
3,minCosto,p05,2476928.5,877167.764843,False,0.570338,0.498180,
4,minCosto,p06,2487781.5,892842.600226,True,0.606141,0.569116,suma_ponderada
5,minCosto,p07,2515684.0,908520.457754,True,0.605169,0.622040,
6,minCosto,p08,2573364.5,924207.042948,False,0.539977,0.618832,
7,minCosto,p09,2589229.5,939882.563504,True,0.564969,0.650462,topsis
8,maxCarbono,p10,2646786.5,955557.431871,True,0.500000,0.617244,
